# Step 03 — Derived Analytical Tables

Four tables the star schema cannot express directly, each requiring window functions or an
ASOF join to build:

| Table | Grain | Why it exists |
|---|---|---|
| `fact_balance_snapshot` | account × month | Semi-additive closing balance, gap-filled through months with no activity |
| `account_distress` | account | First sanction-interest event — a proxy for when trouble started |
| `fact_loan_cohort` | loan | Origination cohort, months on book, balance at origination via ASOF join, affordability ratio |
| `fact_account_behaviour` | account | Recency / frequency / monetary features and product holdings |

**Two limitations are built into this step and must be stated wherever these tables are used.**
They are called out again at the point they arise.


## 3.0 Setup


In [17]:
!pip install -q duckdb


In [18]:
import duckdb, pandas as pd, shutil, os
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/berka-banking-analytics'
PQ   = f'{BASE}/Data/parquet'
OUT  = f'{BASE}/Step03_Derived'
SHARED_DB = f'{BASE}/Data/berka.duckdb'
DB   = '/content/berka.duckdb'

os.makedirs(OUT, exist_ok=True)
shutil.copy(SHARED_DB, DB)
con = duckdb.connect(DB)
print(sorted(con.sql("SELECT table_name FROM duckdb_tables()").df().table_name))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['bridge_disposition', 'dim_account', 'dim_card', 'dim_client', 'dim_date', 'dim_district', 'dim_transaction_type', 'fact_loan', 'fact_order', 'fact_transaction', 'raw_account', 'raw_card', 'raw_client', 'raw_disp', 'raw_district', 'raw_loan', 'raw_order', 'raw_trans']


## 3.1 `fact_balance_snapshot`

The hardest table in the project, and the one that makes the balance measure work in DAX.

An account with no transactions in March still has a balance in March. Aggregating
`fact_transaction` by month would simply omit that account for that month, and every
deposits-over-time chart would then dip for reasons that have nothing to do with money moving.

Three moves solve it:

1. Cross-join every account against every month from its opening date onward — a dense spine.
2. `argmax(balance, ...)` picks the last recorded balance within each month that has activity.
3. `last_value(... IGNORE NULLS)` carries that balance forward across the empty months.

The explicit `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` frame is required. The default
frame is `RANGE`, which would look ahead to peer rows and pull a future balance backwards.

**Limitation 1.** Where several transactions share a date, `trans_id` breaks the tie. Step 01
showed `trans_id` is not a reliable intra-day sequence, so a month-end balance can be off by one
transaction in the rare case that the last day of a month has several. It affects the closing
value only, never the monthly flows.


In [19]:
con.execute("""
CREATE OR REPLACE TABLE fact_balance_snapshot AS
WITH months AS (SELECT DISTINCT year_month, month_start FROM dim_date),
acct_months AS (
  SELECT a.account_key, m.year_month, m.month_start
  FROM dim_account a JOIN months m ON m.month_start >= date_trunc('month', a.opened_date)),
monthly AS (
  SELECT account_key, CAST(date_key/100 AS INTEGER) year_month,
         argmax(balance, CAST(date_key AS BIGINT)*10000000 + trans_id) closing_balance,
         sum(signed_amount) net_flow,
         sum(CASE WHEN signed_amount > 0 THEN signed_amount ELSE 0 END) inflow,
         sum(CASE WHEN signed_amount < 0 THEN -signed_amount ELSE 0 END) outflow,
         count(*) n_transactions
  FROM fact_transaction GROUP BY 1,2)
SELECT am.account_key, am.year_month,
       CAST(strftime(am.month_start,'%Y%m%d') AS INTEGER) month_start_key, am.month_start,
       last_value(m.closing_balance IGNORE NULLS)
         OVER (PARTITION BY am.account_key ORDER BY am.year_month
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) closing_balance,
       coalesce(m.net_flow, 0)       net_flow,
       coalesce(m.inflow, 0)         inflow,
       coalesce(m.outflow, 0)        outflow,
       coalesce(m.n_transactions, 0) n_transactions,
       m.closing_balance IS NULL     is_carried_forward
FROM acct_months am
LEFT JOIN monthly m ON m.account_key = am.account_key AND m.year_month = am.year_month
""")
con.sql("""SELECT count(*) n_rows, count(DISTINCT account_key) n_accounts,
                  min(year_month) lo, max(year_month) hi,
                  count(*) FILTER (WHERE is_carried_forward) carried_forward,
                  count(*) FILTER (WHERE closing_balance IS NULL) unresolved
           FROM fact_balance_snapshot""").df()


,n_rows,n_accounts,lo,hi,carried_forward,unresolved
0,185615,4500,199301,199812,558,0


The snapshot must agree with the transaction fact at every account's final month. If it does
not, the carry-forward has gone wrong.


In [20]:
check = con.sql("""
WITH s AS (SELECT account_key, argmax(closing_balance, year_month) snap_final
           FROM fact_balance_snapshot GROUP BY 1),
     f AS (SELECT account_key, argmax(balance, CAST(date_key AS BIGINT)*10000000+trans_id) fact_final
           FROM fact_transaction GROUP BY 1)
SELECT count(*) n_accounts, count(*) FILTER (WHERE abs(snap_final-fact_final) < 0.01) n_matched
FROM s JOIN f USING(account_key)
""").df()
assert check.n_accounts[0] == check.n_matched[0], 'snapshot does not reconcile to fact_transaction'
check


,n_accounts,n_matched
0,4500,4500


In [21]:
con.sql("""SELECT year_month, count(*) n_accounts, round(sum(closing_balance),0) total_deposits
           FROM fact_balance_snapshot WHERE year_month IN (199312,199412,199512,199612,199712,199812)
           GROUP BY 1 ORDER BY 1""").df()


,year_month,n_accounts,total_deposits
0,199312,1139,37070262.0
1,199412,1578,55335710.0
2,199512,2239,80077945.0
3,199612,3602,130396497.0
4,199712,4500,171285336.0
5,199812,4500,197140234.0


## 3.2 `account_distress`

The dataset records a loan's final status but never when it went bad. Sanction interest
(`SANKC. UROK`) is charged when an account goes into unauthorised debit, so its first occurrence
is the closest thing to a distress date the data contains.

This is a **proxy**, derived from behaviour, not a recorded default date. It is labelled as such
everywhere it appears.


In [22]:
con.execute("""
CREATE OR REPLACE TABLE account_distress AS
SELECT f.account_key, min(f.date_key) first_distress_date_key,
       count(*) n_sanction_events, sum(f.amount) total_sanction_amount
FROM fact_transaction f JOIN dim_transaction_type t USING(trans_type_key)
WHERE t.k_symbol_cz = 'SANKC. UROK'
GROUP BY 1
""")
con.sql("""SELECT l.status_desc, count(*) n_loans, count(d.account_key) n_with_sanction,
                  round(100.0*count(d.account_key)/count(*),1) pct_with_sanction
           FROM fact_loan l LEFT JOIN account_distress d USING(account_key)
           GROUP BY 1 ORDER BY 4 DESC""").df()


,status_desc,n_loans,n_with_sanction,pct_with_sanction
0,"Running, in debt",45,43,95.6
1,"Finished, defaulted",31,29,93.5
2,"Finished, paid",203,5,2.5
3,"Running, OK",403,3,0.7


**Limitation 2 — leakage.** The separation above is extreme, and that is the warning, not the
result. Sanction interest is charged *because* the account is already in trouble, so it is a
symptom of default rather than a predictor of it. It is legitimate for describing what happened
and for a distress-timeline visual. It must be excluded from any model predicting default at
origination, where only information available on the loan date is admissible.


## 3.3 `fact_loan_cohort`

Loans enriched with origination cohort, exposure, and two features that are genuinely known at
origination.

`ASOF LEFT JOIN` attaches each account's balance as at the loan date — an inequality join that
would otherwise need a correlated subquery or a window over a cross join. DuckDB does it in one
clause and it is the single most portfolio-worthy line of SQL in the project.

`payment_to_district_salary` is the affordability proxy. There is no income field, so the
district average salary stands in for it. It is a district-level denominator applied to an
individual, which is an ecological approximation and must be described that way.


In [23]:
con.execute("""
CREATE OR REPLACE TABLE fact_loan_cohort AS
WITH l AS (
  SELECT f.*, d.full_date loan_date, d.year_month cohort_month, d.calendar_year cohort_year
  FROM fact_loan f JOIN dim_date d USING(date_key)),
bal AS (
  SELECT l.loan_id, s.closing_balance balance_at_origination
  FROM l ASOF LEFT JOIN fact_balance_snapshot s
    ON s.account_key = l.account_key AND s.month_start <= l.loan_date)
SELECT l.loan_id, l.account_key, l.date_key, l.loan_date, l.cohort_month, l.cohort_year,
       l.loan_amount, l.duration_months, l.monthly_payment,
       l.status_code, l.status_desc, l.is_finished, l.is_default,
       date_diff('month', l.loan_date, DATE '1998-12-31') months_on_book,
       least(date_diff('month', l.loan_date, DATE '1998-12-31'), l.duration_months) months_observed,
       date_diff('month', l.loan_date, DATE '1998-12-31') >= l.duration_months is_fully_matured,
       b.balance_at_origination,
       round(l.monthly_payment / nullif(dd.avg_salary,0), 3) payment_to_district_salary,
       CASE WHEN dis.first_distress_date_key IS NOT NULL
             AND dis.first_distress_date_key >= l.date_key THEN true ELSE false END distress_after_origination
FROM l
LEFT JOIN bal b USING(loan_id)
LEFT JOIN dim_account a ON a.account_key = l.account_key
LEFT JOIN dim_district dd ON dd.district_key = a.district_key
LEFT JOIN account_distress dis ON dis.account_key = l.account_key
""")
con.sql("""SELECT count(*) n_loans, count(balance_at_origination) with_asof_balance,
                  count(payment_to_district_salary) with_affordability
           FROM fact_loan_cohort""").df()


,n_loans,with_asof_balance,with_affordability
0,682,682,682


In [24]:
con.sql("""
SELECT cohort_year, count(*) n_loans, round(100.0*avg(CAST(is_default AS INT)),1) default_pct,
       round(avg(loan_amount),0) avg_amount, round(avg(months_on_book),1) avg_months_on_book,
       count(*) FILTER (WHERE is_fully_matured) n_matured
FROM fact_loan_cohort GROUP BY 1 ORDER BY 1
""").df()


,cohort_year,n_loans,default_pct,avg_amount,avg_months_on_book,n_matured
0,1993,20,20.0,130964.0,61.8,20
1,1994,101,13.9,132474.0,53.1,85
2,1995,90,13.3,148271.0,41.7,51
3,1996,117,13.7,156561.0,28.8,35
4,1997,196,13.3,156793.0,17.1,43
5,1998,158,2.5,157400.0,6.4,0


**Read that table with care — this is right-censoring, not a lending improvement.** The 1998
cohort shows a much lower default rate simply because those loans have had a few months to fail
rather than several years, and none has reached the end of its term. Cohort default rates are
only comparable at equal months on book. This is exactly why `months_on_book`, `months_observed`
and `is_fully_matured` are stored alongside — so the dashboard can restrict the comparison
rather than present a misleading trend.


In [25]:
con.sql("""
SELECT CASE WHEN payment_to_district_salary < 0.3 THEN 'a. <0.3'
            WHEN payment_to_district_salary < 0.5 THEN 'b. 0.3-0.5'
            WHEN payment_to_district_salary < 0.7 THEN 'c. 0.5-0.7'
            ELSE 'd. 0.7+' END affordability_band,
       count(*) n_loans, round(100.0*avg(CAST(is_default AS INT)),1) default_pct
FROM fact_loan_cohort GROUP BY 1 ORDER BY 1
""").df()


,affordability_band,n_loans,default_pct
0,a. <0.3,212,6.6
1,b. 0.3-0.5,209,8.6
2,c. 0.5-0.7,142,12.7
3,d. 0.7+,119,21.8


## 3.4 `fact_account_behaviour`

One row per account, joining transaction aggregates, snapshot aggregates, product holdings and
loan outcome. This is the table a segmentation page and any predictive model would sit on.

Salary credits are inferred as incoming transfers from another bank with no payment purpose
code, which is the pattern regular income takes in this data. It is a heuristic and named as one.


In [26]:
con.execute("""
CREATE OR REPLACE TABLE fact_account_behaviour AS
WITH tx AS (
  SELECT f.account_key, count(*) n_transactions,
         min(d.full_date) first_txn, max(d.full_date) last_txn,
         sum(CASE WHEN f.signed_amount > 0 THEN f.signed_amount ELSE 0 END) total_inflow,
         sum(CASE WHEN f.signed_amount < 0 THEN -f.signed_amount ELSE 0 END) total_outflow,
         avg(f.balance) avg_balance, min(f.balance) min_balance, max(f.balance) max_balance
  FROM fact_transaction f JOIN dim_date d USING(date_key) GROUP BY 1),
snap AS (
  SELECT account_key, count(*) months_active,
         count(*) FILTER (WHERE closing_balance < 0) months_negative,
         avg(closing_balance) avg_month_end_balance,
         argmax(closing_balance, year_month) final_balance
  FROM fact_balance_snapshot GROUP BY 1),
sal AS (
  SELECT f.account_key, count(*) n_salary_credits, avg(f.amount) avg_salary_credit
  FROM fact_transaction f JOIN dim_transaction_type t USING(trans_type_key)
  WHERE t.operation_cz = 'PREVOD Z UCTU' AND t.k_symbol_cz IS NULL GROUP BY 1),
ord AS (SELECT account_key, count(*) n_standing_orders, sum(amount) standing_order_total
        FROM fact_order GROUP BY 1),
crd AS (SELECT b.account_key, count(*) n_cards, min(c.card_type) card_type
        FROM dim_card c JOIN bridge_disposition b USING(disp_key) GROUP BY 1),
dsp AS (SELECT account_key, count(*) n_users FROM bridge_disposition GROUP BY 1)
SELECT a.account_key, a.district_key, a.opened_date, a.statement_frequency,
       tx.n_transactions, tx.first_txn, tx.last_txn,
       date_diff('day', tx.last_txn, DATE '1998-12-31') recency_days,
       tx.total_inflow, tx.total_outflow, tx.min_balance, tx.max_balance,
       round(tx.total_inflow / nullif(snap.months_active,0), 2) avg_monthly_inflow,
       round(tx.n_transactions * 1.0 / nullif(snap.months_active,0), 2) txns_per_month,
       snap.months_active, snap.months_negative, snap.avg_month_end_balance, snap.final_balance,
       coalesce(sal.n_salary_credits,0) n_salary_credits, sal.avg_salary_credit,
       coalesce(ord.n_standing_orders,0) n_standing_orders,
       coalesce(ord.standing_order_total,0) standing_order_total,
       coalesce(crd.n_cards,0) n_cards, crd.card_type,
       dsp.n_users, dsp.n_users > 1 has_disponent,
       l.loan_id IS NOT NULL has_loan, l.is_default loan_defaulted
FROM dim_account a
LEFT JOIN tx USING(account_key) LEFT JOIN snap USING(account_key)
LEFT JOIN sal USING(account_key) LEFT JOIN ord USING(account_key)
LEFT JOIN crd USING(account_key) LEFT JOIN dsp USING(account_key)
LEFT JOIN fact_loan l USING(account_key)
""")

n = con.sql('SELECT count(*), count(DISTINCT account_key) FROM fact_account_behaviour').fetchone()
assert n[0] == n[1] == 4500, f'grain broken: {n}'
print(f'{n[0]} rows, one per account')


4500 rows, one per account


In [27]:
con.sql("""SELECT count(*) FILTER (WHERE has_loan) with_loan,
                  count(*) FILTER (WHERE n_cards > 0) with_card,
                  count(*) FILTER (WHERE has_disponent) with_disponent,
                  count(*) FILTER (WHERE n_standing_orders > 0) with_standing_order,
                  count(*) FILTER (WHERE months_negative > 0) ever_negative
           FROM fact_account_behaviour""").df()


,with_loan,with_card,with_disponent,with_standing_order,ever_negative
0,682,892,869,3758,193


In [28]:
con.sql("""SELECT loan_defaulted, count(*) n_accounts,
                  round(avg(months_negative),2) avg_months_negative,
                  round(avg(avg_month_end_balance),0) avg_balance,
                  round(avg(avg_monthly_inflow),0) avg_monthly_inflow,
                  round(avg(txns_per_month),1) txns_per_month
           FROM fact_account_behaviour WHERE has_loan GROUP BY 1""").df()


,loan_defaulted,n_accounts,avg_months_negative,avg_balance,avg_monthly_inflow,txns_per_month
0,False,606,0.00,42120.0,29385.0,6.8
1,True,76,2.55,30970.0,28102.0,6.6


## 3.5 Export


In [29]:
DERIVED = ['fact_balance_snapshot','account_distress','fact_loan_cohort','fact_account_behaviour']
for t in DERIVED:
    con.execute(f"COPY {t} TO '{PQ}/{t}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)")

allf = sorted(f for f in os.listdir(PQ) if f.endswith('.parquet'))
sizes = pd.DataFrame([(f, os.path.getsize(f'{PQ}/{f}')) for f in allf], columns=['file','bytes'])
sizes['mb'] = (sizes.bytes/1024**2).round(2)
sizes.to_csv(f'{OUT}/parquet_manifest.csv', index=False)
print(f"{len(allf)} files, {sizes.bytes.sum()/1024**2:.1f} MB total")
sizes.sort_values('bytes', ascending=False)


14 files, 15.9 MB total


,file,bytes,mb
13,fact_transaction.parquet,13313353,12.70
9,fact_balance_snapshot.parquet,2656582,2.53
8,fact_account_behaviour.parquet,287000,0.27
12,fact_order.parquet,112722,0.11
1,bridge_disposition.parquet,86955,0.08
4,dim_client.parquet,72146,0.07
2,dim_account.parquet,53852,0.05
11,fact_loan_cohort.parquet,38241,0.04
5,dim_date.parquet,33560,0.03
10,fact_loan.parquet,21005,0.02


In [30]:
con.close()
shutil.copy(DB, SHARED_DB)
print('derived tables written back →', SHARED_DB)
print(os.listdir(OUT))

derived tables written back → /content/drive/MyDrive/berka-banking-analytics/Data/berka.duckdb
['parquet_manifest.csv', 'berka_derived.duckdb']


---

## Carried into Step 05 (Power BI)

- `fact_balance_snapshot` connects to `dim_date` on `month_start_key` and to `dim_account`.
  Closing balance is **semi-additive**: it sums across accounts but not across time, so the
  measure is `LASTNONBLANKVALUE`, never a plain `SUM`.
- `fact_loan_cohort` replaces `fact_loan` in the model. Same grain, more attributes.
- `fact_account_behaviour` is one row per account and behaves as an extended dimension. Do not
  give it a relationship to `dim_date` — every column is an as-at-1998-12-31 aggregate and
  filtering it by date would be meaningless.
- Any visual comparing default rates across cohorts must be filtered by `months_on_book` or
  restricted to `is_fully_matured`, otherwise it shows censoring and reads as a trend.
- `account_distress` and `distress_after_origination` are for description only, never for a
  model predicting default at origination.
